# Import libraries

In [2]:
import torch
# import tochvision
import torch.nn as nn

# Building the model

In [4]:
#Inception module implementation
class Inception(nn.Module):
    def __init__(self, in_channels, out_1x1, out_3x3_reduce, out_3x3, out_5x5_reduce, out_5x5, out_pool_proj):
        super(Inception, self).__init__()

        # 1x1 conv branch
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, out_1x1, kernel_size=1),
            nn.ReLU()
        )
        # 1x1 conv -> 3x3 conv branch
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, out_3x3_reduce, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(out_3x3_reduce, out_3x3, kernel_size=3, padding=1),
            nn.ReLU()
        )
        # 1x1 conv -> 5x5 conv branch
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, out_5x5_reduce, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(out_5x5_reduce, out_5x5, kernel_size=5, padding=2),
            nn.ReLU()
        )
        # 3x3 max pooling -> 1x1 conv branch
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, out_pool_proj, kernel_size=1),
            nn.ReLU()
        )
def forward(self,x):
    return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)], dim=1)

#GoogLeNet implementation
class GoogLeNet(nn.Module):
    def __init__(self, in_channels, num_classes=1000, dropout_prob=0.4):
        super(GoogLeNet, self).__init__()

        # Initial Layers
        self.initial_layers = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size = 7, stride = 2, padding = 3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size = 3, stride = 2, padding = 1),
            nn.LocalResponseNorm(size = 3),
            nn.Conv2d(64, 64, kernel_size = 1, stride = 1, padding = 0),
            nn.ReLU(),
            nn.Conv2d(64, 192, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.LocalResponseNorm(size = 192)
        )

        self.maxpool = nn.MaxPool2d(kernel_size = 3, stride = 2, padding = 1)

        # Inception Modules
        self.inception3a = Inception(192, 64, 96, 128, 16, 32, 32)
        self.inception3b = Inception(256, 128, 128, 192, 32, 96, 64)
        # self.maxpool = nn.MaxPool2d(kernel_size = 3, stride = 2, padding = 1)

        self.inception4a = Inception(480, 192, 96, 208, 16, 48, 64)
        self.inception4b = Inception(512, 160, 112, 224, 24, 64, 64)
        self.inception4c = Inception(512, 128, 128, 256, 24, 64, 64)
        self.inception4d = Inception(512, 112, 144, 288, 32, 64, 64)
        self.inception4e = Inception(528, 256, 160, 320, 32, 128, 128)
        # self.maxpool = nn.MaxPool2d(kernel_size = 3, stride = 2, padding = 1)
        
        self.inception5a = Inception(832, 256, 160, 320, 32, 128, 128)
        self.inception5b = Inception(832, 384, 192, 384, 48, 128, 128)

        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(p = dropout_prob)
        self.fc = nn.Linear(1024, num_classes)
    
    def forward(self, x):
        x = self.initial_layers(x)
        
        x= self.maxpool(x)

        x = self.inception3a(x)
        x = self.inception3b(x)
        
        x= self.maxpool(x)

        x = self.inception4a(x)
        x = self.inception4b(x)
        x = self.inception4c(x)
        x = self.inception4d(x)
        x = self.inception4e(x)
        
        x= self.maxpool(x)

        x = self.inception5a(x)
        x = self.inception5b(x)
        
        x = self.avg_pool(x)
        
        x = self.dropout(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x    
model=GoogLeNet(in_channels=3, num_classes=1000)
print(model)

GoogLeNet(
  (initial_layers): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (3): LocalResponseNorm(3, alpha=0.0001, beta=0.75, k=1.0)
    (4): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
    (5): ReLU()
    (6): Conv2d(64, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): LocalResponseNorm(192, alpha=0.0001, beta=0.75, k=1.0)
  )
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (inception3a): Inception(
    (branch1): Sequential(
      (0): Conv2d(192, 64, kernel_size=(1, 1), stride=(1, 1))
      (1): ReLU()
    )
    (branch2): Sequential(
      (0): Conv2d(192, 96, kernel_size=(1, 1), stride=(1, 1))
      (1): ReLU()
      (2): Conv2d(96, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (branch3): Sequential(
      (0): Conv2d(192